In [5]:
from langchain_community.document_loaders import PyPDFLoader 
file_path = "C:/Users/zee69/OneDrive/Desktop/Ethical_RAG_Agent/data/raw/veteran_resources.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()
len(docs)

24

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap =100)
chunks = text_splitter.split_documents(docs)
len(chunks)

115

In [7]:
from langchain_community.embeddings import SentenceTransformerEmbeddings 
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 303.75it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
from langchain_community.vectorstores import Chroma 
vector_db = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    collection_name = "veteran_resources",
    persist_directory = "C:/Users/zee69/OneDrive/Desktop/Ethical_RAG_Agent/data/vectordb"
)
vector_db.persist() 

In [9]:
results = vector_db.similarity_search("housing assistance for veterans in chicago", k=3)
for r in results:
    print(r.page_content)
    print("\n---\n") 

Catholic Charities of Chicago Veterans Services 
Includes one HUD senior low income housing with preference for veterans in Hines, IL, 
and rent-subsidized apartments in Chicago, IL.  
https://www.catholiccharities.net/GetHelp/OurServices/VeteransServices.aspx  
Cook County

---

Catholic Charities of Chicago Veterans Services 
Includes one HUD senior low income housing with preference for veterans in Hines, IL, 
and rent-subsidized apartments in Chicago, IL.  
https://www.catholiccharities.net/GetHelp/OurServices/VeteransServices.aspx  
Cook County

---

Catholic Charities of Chicago Veterans Services 
Includes one HUD senior low income housing with preference for veterans in Hines, IL, 
and rent-subsidized apartments in Chicago, IL.  
https://www.catholiccharities.net/GetHelp/OurServices/VeteransServices.aspx  
Cook County

---



In [12]:
from langchain_community.chat_models import ChatOllama
chat_model = ChatOllama(model="llama3.2:3b")

C:\Users\zee69\AppData\Local\Temp\ipykernel_412\2427516969.py:2: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  chat_model = ChatOllama(model="llama3.2:3b")


In [14]:
query = "How can a veteran get housing assistance in Chicago?"
docs = vector_db.similarity_search(query)
context = "\n".join([d.page_content for d in docs])
prompt = f""" 
Use this information to answer the question.

Context: 
{context}

Question: 
{query}
"""
response = chat_model.invoke(prompt)
print(response) 

content='A veteran can get housing assistance in Chicago through the following options:\n\n1. Catholic Charities of Chicago Veterans Services: This organization offers HUD senior low-income housing with preference for veterans, as well as rent-subsidized apartments in Chicago, IL.\n2. Humility Homes & Services: Veterans Housing Program: This program provides shelter, homelessness prevention, and rapid rehousing funds to support veterans and their households.\n\nAdditionally, the Coordinated Entry System in Illinois can provide referrals for local housing projects that participate in the system, which may offer assistance with work clothes or prescription assistance.' additional_kwargs={} response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-02-25T18:44:39.5784687Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 96677959900, 'load_duration': 7692792000, 'prompt_eval_count': 392, 'prompt_eval_duration': 56464123600, 'eval